In [4]:
from pathlib import Path
import os
import sqlite3
import time

from modama_crawler.mime_sql import apply_mime_prefilter_with_resolution

In [5]:
ROOT_PATH = Path("/storage/er-c-data")
DBNAME = "test.db"

In [6]:
con = sqlite3.connect(DBNAME)
cur = con.cursor()

In [5]:
def like_escape(s: str):
    return s.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\\\")

In [6]:
raise Exception()

Exception: 

In [12]:
if False:
    cur.execute("DROP TABLE IF EXISTS crawl")
    cur.execute(
        "CREATE TABLE crawl(path_id INTEGER PRIMARY KEY, path UNIQUE, mtime FLOAT)"
    )
    cur.execute("CREATE INDEX crawl_mtime_index ON crawl(mtime)")
if False:
    cur.execute("DROP TABLE IF EXISTS files")
    cur.execute(
        "CREATE TABLE files(path_id INTEGER PRIMARY KEY, path UNIQUE, mtime FLOAT)"
    )
    cur.execute("CREATE INDEX files_mtime_index ON files(mtime)")
if False:
    cur.execute("DROP TABLE IF EXISTS datasets")
    cur.execute(
        "CREATE TABLE datasets(dataset_id INTEGER PRIMARY KEY, path_id INTEGER UNIQUE, FOREIGN KEY(path_id) REFERENCES files(path_id));"
    )
if False:
    cur.execute("DROP TABLE IF EXISTS mimes")
    cur.execute(
        "CREATE TABLE mimes(dataset_id INTEGER, mime TEXT, FOREIGN KEY(dataset_id) REFERENCES datasets(dataset_id), UNIQUE (dataset_id, mime));"
    )
if False:
    cur.execute("DROP TABLE IF EXISTS files_in_datasets")
    cur.execute(
        "CREATE TABLE files_in_datasets(dataset_id INTEGER, path_id INTEGER, FOREIGN KEY(dataset_id) REFERENCES datasets(dataset_id), FOREIGN KEY(path_id) REFERENCES files(path_id), UNIQUE(dataset_id, path_id));"
    )

In [13]:
def maybe_path_stat(path):
    try:
        return (str(path), os.stat(path, follow_symlinks=False))
    except PermissionError, FileNotFoundError:
        return (str(path), None)


def latest_update(stat):
    return max(stat.st_mtime, stat.st_ctime)

In [14]:
t = time.monotonic()


def maybe_update(s):
    global t
    t2 = time.monotonic()
    if t2 - t > 1:
        print(s)
        t = t2

In [17]:
%%time
if False:
    for root, dirs, files in os.walk(ROOT_PATH, followlinks=False):
        path_stat = (maybe_path_stat(os.path.join(root, d)) for d in dirs)
        db_updates = list(
            (str(item[0]), latest_update(item[1]), latest_update(item[1]))
            for item in path_stat
            if item[1] is not None
        )
        cur.executemany(
            "INSERT INTO crawl(path, mtime) VALUES(?, ?) ON CONFLICT(path) DO UPDATE SET mtime = ?;",
            db_updates,
        )
        path_stat = (maybe_path_stat(os.path.join(root, d)) for d in files)
        db_updates = list(
            (str(item[0]), latest_update(item[1]), latest_update(item[1]))
            for item in path_stat
            if item[1] is not None
        )
        cur.executemany(
            "INSERT INTO crawl(path, mtime) VALUES(?, ?) ON CONFLICT(path) DO UPDATE SET mtime = ?;",
            db_updates,
        )
        maybe_update(root)
        con.commit()

/storage/er-c-data
/storage/er-c-data/adhoc/in_situ_diff/2022-08-03-annealing/raw/annealing_gold/shift_calibration/0001
/storage/er-c-data/adhoc/desyptycho/data_for_paper/raw/live/scan_00063/pilctriggergenerator_04
/storage/er-c-data/adhoc/phonon_eels/Phonon EELS project/Code/Display_Hn0/Trunk/.svn/pristine/4d
/storage/er-c-data/adhoc/soft_matter/FH_Aachen/Winter_Membran/Images_Paper
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_afe/output/afe.save/K00164
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_afe/output/afe.save/K01135
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_2fe/afe.save/K00001
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_2fe/afe.save/K00980
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_2fe/afe.save/K01957
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_m/M-scf.save
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_m/M-scf.save/K00861
/storage/er-c-data/adhoc/alsctizr/du/du/qe/hfo2/scf_m/M-scf.save/K01833
/storage/er-c-data/adhoc/alsct

In [19]:
if False:
    cur.execute(
        "INSERT INTO files (path, mtime) SELECT path, mtime FROM crawl WHERE TRUE ON CONFLICT (path) DO UPDATE SET mtime = excluded.mtime WHERE excluded.mtime != files.mtime;"
    )

In [20]:
con.commit()

In [21]:
if False:
    res = cur.execute(
        "SELECT path FROM files WHERE path LIKE '%/miniforge3/envs' OR path LIKE '%/miniconda3/envs' OR path LIKE '%/.conda/envs' ORDER BY LENGTH(path) ASC;"
    )
    conda_envs = res.fetchall()

    for (envfolder,) in conda_envs:
        p = Path(envfolder)
        if p.is_dir() and not p.is_symlink():
            for env in p.iterdir():
                if (
                    env.is_dir()
                    and not env.is_symlink()
                    and (env / "conda-meta").is_dir()
                ):
                    path_res = cur.execute(
                        "SELECT path_id FROM files WHERE path == ? ;", (str(env),)
                    )
                    path_ids = path_res.fetchall()
                    if not path_ids:
                        continue
                    elif len(path_ids) > 1:
                        raise RuntimeError("WTF")
                    else:
                        path_id = path_ids[0][0]
                        cur.execute(
                            "INSERT INTO datasets(path_id) VALUES (?) ON CONFLICT DO NOTHING;",
                            (path_id,),
                        )
                        res = cur.execute(
                            "SELECT dataset_id FROM datasets WHERE path_id == ?;",
                            (path_id,),
                        )
                        dataset_ids = res.fetchall()
                        if len(dataset_ids) != 1:
                            raise RuntimeError("WTF")
                        dataset_id = dataset_ids[0][0]
                        cur.executemany(
                            "INSERT OR IGNORE INTO mimes(dataset_id, mime) VALUES (?, ?);",
                            (
                                (dataset_id, mime)
                                for mime in (
                                    "application/x.conda-env",
                                    "inode/directory",
                                )
                            ),
                        )
                        cur.execute(
                            "INSERT INTO files_in_datasets (dataset_id, path_id) SELECT ?, path_id FROM files WHERE path LIKE ? ESCAPE '\\' AND path_id NOT IN ( SELECT path_id FROM files_in_datasets ) ON CONFLICT DO NOTHING;",
                            (
                                dataset_id,
                                like_escape(str(env)) + "/%",
                            ),
                        )

In [24]:
res = cur.execute(
    "SELECT path FROM files WHERE path LIKE '%/envs' ORDER BY LENGTH(path) ASC;"
)
res.fetchall()

[('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/envs',)]

In [1]:
if False:
    res = cur.execute(
        "SELECT path FROM files WHERE path LIKE '%node_modules' ORDER BY LENGTH(path) ASC;"
    )
    node_modules = res.fetchall()

    for (node_module,) in node_modules:
        p = Path(node_module)
        # node_modules contains many node_modules
        if p.is_dir() and not p.is_symlink() and "node_modules" not in p.parts[:-1]:
            path_res = cur.execute(
                "SELECT path_id FROM files WHERE path == ? ;", (str(node_module),)
            )
            path_ids = path_res.fetchall()
            if not path_ids:
                continue
            elif len(path_ids) > 1:
                raise RuntimeError("WTF")
            else:
                path_id = path_ids[0][0]
                cur.execute(
                    "INSERT INTO datasets(path_id) VALUES (?) ON CONFLICT DO NOTHING;",
                    (path_id,),
                )
                res = cur.execute(
                    "SELECT dataset_id FROM datasets WHERE path_id == ?;", (path_id,)
                )
                dataset_ids = res.fetchall()
                if len(dataset_ids) != 1:
                    raise RuntimeError("WTF")
                dataset_id = dataset_ids[0][0]
                cur.executemany(
                    "INSERT OR IGNORE INTO mimes(dataset_id, mime) VALUES (?, ?);",
                    (
                        (dataset_id, mime)
                        for mime in ("application/x.node-modules", "inode/directory")
                    ),
                )
                cur.execute(
                    "INSERT INTO files_in_datasets (dataset_id, path_id) SELECT ?, path_id FROM files WHERE path LIKE ? ESCAPE '\\' AND path_id NOT IN ( SELECT path_id FROM files_in_datasets ) ON CONFLICT DO NOTHING;",
                    (
                        dataset_id,
                        like_escape(str(node_module)) + "/%",
                    ),
                )

            print(node_module)

In [2]:
if False:
    apply_mime_prefilter_with_resolution(
        conn=con,
        files_table="files",
        datasets_table="datasets",
        mimes_table="mimes",
        files_in_datasets_table="files_in_datasets",
    )

In [11]:
res = cur.execute(
    "SELECT datasets.dataset_id, mime, files.path FROM mimes JOIN datasets ON mimes.dataset_id == datasets.dataset_id JOIN files on datasets.path_id == files.path_id WHERE mimes.mime != 'text/html' LIMIT 100 OFFSET 100;"
)
cur.fetchall()

[(530,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/asyncio/transports.py'),
 (531,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/asyncio/trsock.py'),
 (532,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/asyncio/unix_events.py'),
 (533,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/asyncio/windows_events.py'),
 (534,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/asyncio/windows_utils.py'),
 (535,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/base64.py'),
 (536,
  'text/x-python',
  '/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/lib/python3.12/bdb.py'),
 (537,
  'text/x-python',
  '/storage/er-c-data/adhoc/aid

In [10]:
con.commit()

In [220]:
cur.executemany(
    "INSERT OR IGNORE INTO temp.mimes(dataset_id, mime) VALUES (?, ?);",
    ((2, mime) for mime in ("application/node-modules",)),
)

In [221]:
res = cur.execute(
    "SELECT datasets.dataset_id, mime FROM temp.mimes JOIN datasets ON datasets.dataset_id = temp.mimes.dataset_id;"
)
res.fetchall()

[(2, 'application/node-modules')]

In [172]:
if False:
    cur.execute("DELETE FROM mimes;")

In [178]:
res = cur.execute("SELECT dataset_id, mime FROM mimes;")
res.fetchall()

[(1, 'application/octet-stream'),
 (1, 'text/plain'),
 (2, 'application/node-modules')]

In [193]:
%%time
res = cur.execute(
    "SELECT path_id from files WHERE path_id NOT IN ( SELECT path_id FROM files_in_datasets) ORDER BY path_id ASC;"
)
# res.fetchall()

CPU times: user 11.8 ms, sys: 0 ns, total: 11.8 ms
Wall time: 10.6 ms


In [194]:
from modama_crawler import mime_sql

In [195]:
filters = mime_sql.collect_sql_path_filters()

In [198]:
case_sensitive = [f for f in filters if f.case_sensitive]

In [200]:
mime_sql.build_sql_where_clause(filters=case_sensitive[:2])

("((path LIKE ? ESCAPE '\\') OR (path LIKE ? ESCAPE '\\')) OR ((path LIKE ? ESCAPE '\\') OR (path LIKE ? ESCAPE '\\'))",
 ('%/%.gs', '%.gs', '%/%.C', '%.C'))

In [187]:
res = cur.execute("SELECT DISTINCT path from files WHERE path_id == 266 ;")
res.fetchall()

[('/storage/er-c-data/adhoc/datamanagement/2022-03-22 my first session/raw/bydieter.txt',)]

In [207]:
res = cur.execute("SELECT path, ? from files WHERE path_id == 266 ;", (23,))
res.fetchall()

[('/storage/er-c-data/adhoc/datamanagement/2022-03-22 my first session/raw/bydieter.txt',
  23)]

In [78]:
%%time
res = cur.execute(
    "SELECT ? , path, mtime FROM files WHERE mtime > ? and mtime < ? ORDER BY mtime DESC",
    (23, time.time() - 9 * 60 * 60 * 24, time.time() - 0 * 60 * 60 * 24),
)
res.fetchall()

CPU times: user 2.24 ms, sys: 621 μs, total: 2.86 ms
Wall time: 2.2 ms


[(23,
  '/storage/er-c-data/adhoc/remade_30820/remade30820_STEM/02@15K_undef_20260206FIB/20260320_Spectra/20260320_1110_10.0_Mx_STEM_HAADF.emd',
  1783439822.3839767),
 (23,
  '/storage/er-c-data/adhoc/remade_30820/remade30820_STEM/02@15K_undef_20260206FIB/20260320_Spectra/20260320_1110_10.0_Mx_STEM_HAADF Gaussian blur(HAADF).tif',
  1783436437.1227224),
 (23,
  '/storage/er-c-data/adhoc/remade_30820/remade30820_STEM/02@15K_undef_20260206FIB/20260320_Spectra',
  1783436436.9591174),
 (23,
  '/storage/er-c-data/adhoc/remade_30820/remade30820_STEM/02@15K_undef_20260206FIB/20260320_Spectra/20260320_1529_10.0_Mx_STEM_HAADF.emd',
  1783436330.542095),
 (23,
  '/storage/er-c-data/adhoc/remade_30820/02_B2@15K_undeformed/Thumbs.db',
  1783436123.2767377),
 (23,
  '/storage/er-c-data/adhoc/weber/Retreat_Sammelantrag_Pendeln.xlsx',
  1783432113.2788804),
 (23, '/storage/er-c-data/adhoc/weber', 1783432113.2158785),
 (23,
  '/storage/er-c-data/adhoc/weber/Retreat_Sammelantrag_Pendeln.pdf',
  17834

In [ ]:
res = cur.execute(
    "SELECT path, mtime FROM files WHERE path = ? ORDER BY mtime DESC",
    ("/storage/er-c-data/adhoc/libertem/media/2026-06 DECTRIS visit",),
)
res.fetchall()

In [205]:
%%time
res = cur.execute(
    "SELECT path, mtime FROM files WHERE path LIKE '%/storage/er-c-data%/lib/python_.__' ORDER BY mtime DESC"
)
res.fetchall()

CPU times: user 261 ms, sys: 75.8 ms, total: 337 ms
Wall time: 336 ms


[('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/zstandard-0.25.0-py312hd41f8a7_1/lib/python3.12',
  1775724144.7253797),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/ruamel.yaml.clib-0.2.14-py312hcd1a082_0/lib/python3.12',
  1775724131.3999858),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/ruamel.yaml-0.18.16-py312hcd1a082_0/lib/python3.12',
  1775724131.003974),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/python-3.13.12-h4c0d347_100_cp313/lib/python3.13',
  1775724123.8867638),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/python-3.12.12-h91f4b29_1_cpython/lib/python3.12',
  1775724114.1044745),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/pycosat-0.6.6-py312hcd1a082_3/lib/python3.12',
  1775724111.0443842),
 ('/storage/er-c-data/adhoc/aidenoise/grace-cachedata-backup/miniforge3/pkgs/msgpack-python-

In [14]:
cur.execute("SELECT sqlite_version();")
print(cur.fetchone())

('3.53.2',)
